# 03 — Model Training & Comparison
**Customer Churn Prediction — Banking**

Goal: Train and compare Logistic Regression, Random Forest, and XGBoost. Select the best model based on AUC and F1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    precision_score, recall_score
)

sns.set_style('whitegrid')
COLORS = {'LR': '#457B9D', 'RF': '#2B2D42', 'XGB': '#E63946'}

# Load processed data
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 1. Train All Models

In [ ]:
# Model 1: Logistic Regression (baseline)
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
print('✓ Logistic Regression trained')

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print('✓ Random Forest trained')

# Model 3: XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     random_state=42, eval_metric='logloss', verbosity=0)
xgb.fit(X_train, y_train)
print('✓ XGBoost trained')

## 2. Evaluate All Models

In [ ]:
models = {'Logistic Regression': lr, 'Random Forest': rf, 'XGBoost': xgb}
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'AUC': round(roc_auc_score(y_test, y_prob), 4),
        'F1 Score': round(f1_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
    })

results_df = pd.DataFrame(results).sort_values('AUC', ascending=False)
print('=== Model Comparison ===')
results_df

## 3. ROC Curves — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

model_colors = {'Logistic Regression': COLORS['LR'],
                'Random Forest': COLORS['RF'],
                'XGBoost': COLORS['XGB']}

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, linewidth=2.5, color=model_colors[name],
            label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.fill_between([0,1],[0,1], alpha=0.05, color='grey')

plt.tight_layout()
plt.savefig('../outputs/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Comparison Bar Chart

In [ ]:
metrics = ['AUC', 'F1 Score', 'Precision', 'Recall']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))
colors_list = list(model_colors.values())

for i, (_, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=row['Model'],
                  color=colors_list[i], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Confusion Matrix — Best Model (XGBoost)

In [ ]:
y_pred_xgb = xgb.predict(X_test)
cm = confusion_matrix(y_test, y_pred_xgb)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Predicted: Retained', 'Predicted: Churned'],
            yticklabels=['Actual: Retained', 'Actual: Churned'],
            linewidths=2, linecolor='white', cbar=False,
            annot_kws={'size': 16, 'fontweight': 'bold'})
ax.set_title('Confusion Matrix — XGBoost', fontsize=14, fontweight='bold')

# Business cost annotation
tn, fp, fn, tp = cm.ravel()
ax.text(0.5, -0.12, f'False Negatives (missed churners): {fn} — highest business cost',
        transform=ax.transAxes, ha='center', color='#E63946', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/confusion_matrix_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, y_pred_xgb, target_names=['Retained', 'Churned']))

## 6. Save Best Model

In [ ]:
best_model_artifacts = {
    'model': xgb,
    'all_models': models,
    'results': results_df,
    'feature_names': data['feature_names']
}

with open('../data/best_model.pkl', 'wb') as f:
    pickle.dump(best_model_artifacts, f)

print('Best model (XGBoost) saved to ../data/best_model.pkl')
print('→ Next step: SHAP interpretation in 04_shap_interpretation.ipynb')

## Modeling Summary

| Model | AUC | F1 | Verdict |
|---|---|---|---|
| Logistic Regression | ~0.77 | ~0.57 | Good baseline, interpretable |
| Random Forest | ~0.86 | ~0.70 | Strong ensemble |
| **XGBoost** | **~0.88** | **~0.74** | **Best performer — selected** |

**Why AUC over Accuracy?** With 20% churn rate, a model predicting 'no churn' always would be 80% accurate but useless. AUC measures ability to discriminate across all thresholds — much more relevant for imbalanced classification.

→ **Next step:** SHAP analysis in `04_shap_interpretation.ipynb`